# Revisión de AUDITORÍA y trazabilidad

Cada capa registra su ejecución y enlaza con el ID de la capa anterior:
`bronce.audit_id ← silver.bronze_audit_id` y `silver.audit_id ← gold.silver_audit_id`.
Esto permite rastrear cualquier tabla gold hasta la descarga original.

In [1]:
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq

DATA = Path("../data") if Path("../data").exists() else Path("data")
CATS = ["green", "yellow", "fhv", "fhvhv"]
YEARS = [2023, 2024, 2025]
MESES = [(y, m) for y in YEARS for m in range(1, 13)]
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(25)


def parquet_rows(path: Path) -> int | None:
    """Filas totales de un archivo/directorio parquet leyendo solo metadatos.

    Ojo: Spark escribe "archivos" mensuales como DIRECTORIOS llamados *.parquet
    con part-files dentro — por eso se filtra con is_file(). Se ignora la
    basura _temporary y los archivos de 0 bytes que deja un job interrumpido.
    """
    if not path.exists():
        return None
    files = (
        [path]
        if path.is_file()
        else [
            f for f in path.rglob("*.parquet")
            if f.is_file() and f.stat().st_size > 0 and "_temporary" not in str(f)
        ]
    )
    if not files:
        return None
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)


def dir_mb(path: Path) -> float:
    """Peso en MB de un archivo o directorio."""
    if not path.exists():
        return 0.0
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return round(sum(f.stat().st_size for f in files) / 1024**2, 2)

## Las tres auditorías

In [2]:
audit_b = pl.read_parquet(DATA / "bronze" / "audit.parquet")
audit_s = pl.read_parquet(DATA / "silver" / "audit.parquet")
audit_g = pl.read_parquet(DATA / "gold" / "audit.parquet")
print(f"bronce: {audit_b.height} registros | silver: {audit_s.height} | gold: {audit_g.height}")
audit_g.select("mart_name", "mode", "rowcount_output", "end_timestamp").tail(9)

bronce: 156 registros | silver: 146 | gold: 30


mart_name,mode,rowcount_output,end_timestamp
str,str,i64,str
"""mart_demand_volume""","""incremental""",82738,"""2026-07-16T20:49:59.070086"""
"""mart_financial_performance""","""incremental""",25842,"""2026-07-16T20:50:03.435809"""
"""mart_operational_profile""","""incremental""",25842,"""2026-07-16T20:50:06.290398"""
"""mart_supply_demand_balance""","""incremental""",327404,"""2026-07-16T20:50:12.233690"""
"""mart_abc_xyz_zones""","""incremental""",255,"""2026-07-16T20:50:15.101160"""
"""mart_tipping_behavior""","""incremental""",4755,"""2026-07-16T20:50:18.178746"""
"""ml_feat_arima_trips""","""incremental""",4040,"""2026-07-16T20:50:20.498278"""
"""ml_feat_kmodes_trips""","""incremental""",2349190,"""2026-07-16T20:50:32.052504"""
"""ml_feat_isolation_fraud""","""incremental""",2349190,"""2026-07-16T20:50:43.402122"""


## Verificación de la cadena de trazabilidad

In [3]:
ids_bronce = set(audit_b["audit_id"].to_list())
ids_silver = set(audit_s["audit_id"].to_list())

silver_enlazado = audit_s.filter(pl.col("bronze_audit_id").is_in(list(ids_bronce))).height
gold_enlazado = audit_g.filter(pl.col("silver_audit_id").is_in(list(ids_silver))).height
print(f"registros silver que enlazan a un bronce valido: {silver_enlazado}/{audit_s.height}")
print(f"registros gold que enlazan a un silver valido:   {gold_enlazado}/{audit_g.height}")
print(f"CADENA COMPLETA: {silver_enlazado == audit_s.height and gold_enlazado == audit_g.height}")

registros silver que enlazan a un bronce valido: 146/146
registros gold que enlazan a un silver valido:   30/30
CADENA COMPLETA: True


## Conteos de la corrida vigente de silver (calidad)

La auditoría acumula historia (re-corridas, reparaciones); se toma la
**última entrada por archivo** para reflejar el estado vigente.

In [4]:
ultima = (audit_s.filter(pl.col("rowcount_quality").is_not_null())
    .sort("end_timestamp", descending=True)
    .unique(subset=["source_file"], keep="first"))
resumen = ultima.select(
    pl.col("rowcount_bronze").sum().alias("bronce"),
    pl.col("rowcount_quality").sum().alias("limpios"),
    pl.col("rowcount_quarantined").sum().alias("rechazados"),
)
print(f"archivos con auditoria vigente: {ultima.height}")
print(resumen)
print(f"bronce == limpios + rechazados: {resumen[0,0] == resumen[0,1] + resumen[0,2]}")

archivos con auditoria vigente: 146
shape: (1, 3)
┌───────────┬───────────┬────────────┐
│ bronce    ┆ limpios   ┆ rechazados │
│ ---       ┆ ---       ┆ ---        │
│ i64       ┆ i64       ┆ i64        │
╞═══════════╪═══════════╪════════════╡
│ 911452617 ┆ 843099229 ┆ 68353388   │
└───────────┴───────────┴────────────┘
bronce == limpios + rechazados: True
